In [1]:
import gurobipy as gp
from gurobipy import GRB, nlfunc
import math

In [2]:

# Build a minimal model capturing just the IIS block for state (1,1,0), node 41
m = gp.Model("ESF_min_test")

# Design variables (fixed to 0)
d0 = m.addVar(lb=0.0, ub=0.0, name="d[0]")
d1 = m.addVar(lb=0.0, ub=0.0, name="d[1]")
# d2 is not used in this snippet, so we omit it here

# Theta, jac, prod
theta0 = m.addVar(lb=-GRB.INFINITY, name="sf_(1,_1,_0)_theta[0,41]")
theta1 = m.addVar(lb=-GRB.INFINITY, name="sf_(1,_1,_0)_theta[1,41]")
jac    = m.addVar(lb=-GRB.INFINITY, name="sf_(1,_1,_0)_jac[41]")
prod   = m.addVar(lb=-GRB.INFINITY, name="sf_(1,_1,_0)_prod[41]")

# Region selector binaries
z10 = m.addVar(vtype=GRB.BINARY, name="sf_(1,_1,_0)_z[1,0,41]")
z11 = m.addVar(vtype=GRB.BINARY, name="sf_(1,_1,_0)_z[1,1,41]")
z12 = m.addVar(vtype=GRB.BINARY, name="sf_(1,_1,_0)_z[1,2,41]")
z13 = m.addVar(vtype=GRB.BINARY, name="sf_(1,_1,_0)_z[1,3,41]")

bigM = 1e4

Set parameter Username
Academic license - for non-commercial use only - expires 2027-02-12


In [3]:
# Region selection
m.addConstr(z10 + z11 + z12 + z13 == 1,
            name="sf_(1,_1,_0)_region_select[1,41]")

# Region inequalities (using your IIS coefficients)
m.addConstr(
    -0.7071067811865475 * d0
    + 0.7071067811865477 * theta0
    + bigM * z10 <= bigM,
    name="sf_(1,_1,_0)_region_ineq[1,0,41,1]"
)

m.addConstr(
    -0.7359310117618293 * d1
    + 0.6770565308208838 * theta0
    + bigM * z11 <= bigM,
    name="sf_(1,_1,_0)_region_ineq[1,1,41,2]"
)

m.addConstr(
    -d0 + bigM * z12 <= 9996.16368286445,
    name="sf_(1,_1,_0)_region_ineq[1,2,41,3]"
)

m.addConstr(
    -d0 + bigM * z13 <= 9996.16368286445,
    name="sf_(1,_1,_0)_region_ineq[1,3,41,3]"
)

<gurobi.Constr *Awaiting Model Update*>

In [4]:
# Nonlinear pdf constraint: prod = jac * pdf(theta0, theta1)
# This mirrors your pdf_builder
eps = 1e-6
S = theta0
D = theta1
x = S - 8.0

pdf_expr = (1.0 / (1.2 * math.pi)) * (1.0 / (x + eps)) * nlfunc.exp(
    -1.39 * nlfunc.log(x + eps) * nlfunc.log(x + eps)
    - 0.5 * (D - 7.0) * (D - 7.0)
)

m.addConstr(prod == jac * pdf_expr, name="sf_(1,_1,_0)_prod_def[41]")

<gurobi.GenConstr *Awaiting Model Update*>

In [5]:

# Dummy objective
m.setObjective(d0 + d1, GRB.MINIMIZE)

# Enable nonlinear
m.Params.FuncNonlinear = 1

Set parameter FuncNonlinear to value 1


In [6]:
m.optimize()

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13700, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 24 logical processors, using up to 24 threads

Optimize a model with 5 rows, 10 columns and 14 nonzeros
Model fingerprint: 0x3c5d6c35
Model has 1 general nonlinear constraint (7 nonlinear terms)
Variable types: 6 continuous, 4 integer (4 binary)
Coefficient statistics:
  Matrix range     [7e-01, 1e+04]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+04]
Presolve removed 3 rows and 5 columns
Presolve time: 0.00s

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 24 available processors)

Solution count 0

Model is infeasible
Best objective -, best bound -, gap -


In [7]:
print("Status:", m.Status)
if m.Status == GRB.OPTIMAL:
    print("Feasible minimal model found.")
    print("d0 =", d0.X, "d1 =", d1.X)
    print("theta0 =", theta0.X, "theta1 =", theta1.X)
    print("z10,z11,z12,z13 =", z10.X, z11.X, z12.X, z13.X)
elif m.Status == GRB.INFEASIBLE:
    print("Minimal model is infeasible.")

Status: 3
Minimal model is infeasible.
